In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time

# 대전시 행사안내 월간 일정표 엔드포인트
base_url = "https://www.daejeon.go.kr/fvu/FvuEventDayScheduleList.do"

# 수집할 연도 설정 (2023년 ~ 2026년)
target_years = [2023, 2024, 2025, 2026]
menu_seq = "7419"  # 제공해주신 URL의 menuSeq 값

all_events = []

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("대전시 행사 일정 크롤링을 시작합니다...")

for year in target_years:
    for month in range(1, 13):
        # 2026년의 경우 미래 데이터나 불필요한 기간은 조절 가능
        if year == 2026 and month > 12:
            break
            
        params = {
            "year": str(year),
            "month": str(month),
            "menuSeq": menu_seq
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers)
            if response.status_code != 200:
                print(f"[{year}년 {month}월] 요청 실패 (상태 코드: {response.status_code})")
                continue
                
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # [주의] 실제 브라우저(F12)에서 행사 목록이 담긴 HTML 태그 구조를 확인 후 
            # 아래 select 경로(.table 등)를 알맞게 수정하셔야 합니다.
            event_rows = soup.select("table tbody tr") # 예시: 테이블 행
            
            count = 0
            for row in event_rows:
                # 각 행에서 텍스트 데이터 혹은 a 태그 추출
                cols = row.find_all(['th', 'td'])
                if len(cols) > 1:
                    row_texts = [col.get_text(strip=True) for col in cols]
                    
                    # 상세 페이지 이동 함수(fn_detailView)가 걸려있는지 확인
                    link_tag = row.find("a")
                    detail_link = ""
                    if link_tag and 'onclick' in link_tag.attrs:
                        detail_link = link_tag['onclick']
                        
                    all_events.append({
                        "year": year,
                        "month": month,
                        "data": row_texts,
                        "script_action": detail_link
                    })
                    count += 1
            
            print(f"수집 성공: {year}년 {month}월 (총 {count}개 행사)")
            
            # 서버 부하 방지를 위한 0.5초 대기
            time.sleep(0.5)
            
        except Exception as e:
            print(f"에러 발생 [{year}년 {month}월]: {e}")

# 수집된 데이터를 JSON 파일로 저장
with open('daejeon_event_schedule.json', 'w', encoding='utf-8') as f:
    json.dump(all_events, f, ensure_ascii=False, indent=4)

print(f"총 {len(all_events)}개의 일정이 'daejeon_event_schedule.json' 파일로 안전하게 저장되었습니다.")

대전시 행사 일정 크롤링을 시작합니다...
수집 성공: 2023년 1월 (총 36개 행사)
수집 성공: 2023년 2월 (총 33개 행사)
수집 성공: 2023년 3월 (총 36개 행사)
수집 성공: 2023년 4월 (총 36개 행사)
수집 성공: 2023년 5월 (총 36개 행사)
수집 성공: 2023년 6월 (총 35개 행사)
수집 성공: 2023년 7월 (총 37개 행사)
수집 성공: 2023년 8월 (총 36개 행사)
수집 성공: 2023년 9월 (총 35개 행사)
수집 성공: 2023년 10월 (총 36개 행사)
수집 성공: 2023년 11월 (총 35개 행사)
수집 성공: 2023년 12월 (총 37개 행사)
수집 성공: 2024년 1월 (총 36개 행사)
수집 성공: 2024년 2월 (총 34개 행사)
수집 성공: 2024년 3월 (총 37개 행사)
수집 성공: 2024년 4월 (총 35개 행사)
수집 성공: 2024년 5월 (총 36개 행사)
수집 성공: 2024년 6월 (총 36개 행사)
수집 성공: 2024년 7월 (총 36개 행사)
수집 성공: 2024년 8월 (총 36개 행사)
수집 성공: 2024년 9월 (총 35개 행사)
수집 성공: 2024년 10월 (총 36개 행사)
수집 성공: 2024년 11월 (총 35개 행사)
수집 성공: 2024년 12월 (총 36개 행사)
수집 성공: 2025년 1월 (총 36개 행사)
수집 성공: 2025년 2월 (총 33개 행사)
수집 성공: 2025년 3월 (총 37개 행사)
수집 성공: 2025년 4월 (총 35개 행사)
수집 성공: 2025년 5월 (총 36개 행사)
수집 성공: 2025년 6월 (총 35개 행사)
수집 성공: 2025년 7월 (총 36개 행사)
수집 성공: 2025년 8월 (총 37개 행사)
수집 성공: 2025년 9월 (총 35개 행사)
수집 성공: 2025년 10월 (총 36개 행사)
수집 성공: 2025년 11월 (총 36개 행사)
수집 성공: 2025년 12월 (총 36개